# Bike-Sharing Demand Investigation
## How do time, season, weather, and environment influence hourly demand?

**Research question:** How do time of day, season, weather, and other environmental
factors influence hourly bike-sharing demand, and can we build a model that reliably
predicts it?

**Dataset:** UCI Bike Sharing Dataset (Fanaee-T & Gama, 2013) — `hour.csv`  
**Target variable:** `cnt` — total hourly bike rentals (casual + registered)  
**Period covered:** 2011–2012, Washington D.C. Capital Bikeshare system

---

### How to run this notebook

This notebook is designed to execute **top to bottom** in Google Colab or any
Jupyter-compatible environment. All data are downloaded automatically; no local files
are needed.

> **GPU is not required.** All models use CPU-based scikit-learn.

---
## Section 1 — Environment Setup

Install and import all required libraries. In Google Colab, most of these are
pre-installed; this cell ensures exact versions are available.

In [ ]:
# Install required libraries (safe to run in Colab or locally)
# !pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
print('Libraries loaded successfully.')

In [ ]:
# ---------------------------------------------------------------------------
# Dataset Acquisition
# Primary method: download from UCI Machine Learning Repository
# Fallback: upload from your local machine (uncomment the block below)
# ---------------------------------------------------------------------------

import urllib.request
import zipfile
import io

DATASET_URL = (
    'https://archive.ics.uci.edu/ml/machine-learning-databases/'
    '00275/Bike-Sharing-Dataset.zip'
)


def load_hour_csv_from_uci(url):
    """Download the UCI Bike Sharing Dataset zip and return the hour.csv DataFrame."""
    print(f'Downloading dataset from UCI...')
    with urllib.request.urlopen(url, timeout=30) as response:
        zip_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        print('Zip contents:', zf.namelist())
        with zf.open('hour.csv') as f:
            df = pd.read_csv(f)
    return df


try:
    df = load_hour_csv_from_uci(DATASET_URL)
    print(f'Dataset loaded successfully: {df.shape[0]:,} rows x {df.shape[1]} columns')
except Exception as e:
    print(f'Download failed: {e}')
    print()
    print('FALLBACK  Upload hour.csv from your local machine:')
    print('  1. Uncomment the three lines below')
    print('  2. Re-run this cell')
    print('  3. Use the file picker to select hour.csv')
    print()
    # Uncomment to use the Colab file-upload widget:
    # from google.colab import files
    # uploaded = files.upload()          # select hour.csv when prompted
    # df = pd.read_csv(list(uploaded.keys())[0])
    raise

In [ ]:
print('Shape:', df.shape)
print()
print('First 3 rows:')
df.head(3)

---
## Section 2 — Data Exploration

Before modelling we inspect the dataset structure, verify data quality, and build
intuition about the target and predictors.

In [ ]:
print('=== Dataset Overview ===')
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print()
print('Column dtypes:')
print(df.dtypes)
print()
print('Descriptive statistics for numeric columns:')
df.describe().round(3)

In [ ]:
print('=== Missing Values and Duplicates ===')
missing = df.isnull().sum()
if missing.sum() > 0:
    print('Missing values per column:')
    print(missing[missing > 0])
else:
    print('Missing values: None  dataset is complete.')
print()
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')

### Target Variable: `cnt`

`cnt` is the total number of bike rentals in a given hour. It equals `casual + registered`.
We exclude `casual` and `registered` as predictors because they would cause target leakage
--- together they sum exactly to `cnt`.

The distribution is right-skewed (skewness ≈ 1.28): most hours have moderate demand,
but a tail of peak hours inflates the mean above the median.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df['cnt'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['cnt'].mean(), color='firebrick', linewidth=1.5, linestyle='--',
                label=f"Mean = {df['cnt'].mean():.1f}")
axes[0].axvline(df['cnt'].median(), color='darkorange', linewidth=1.5, linestyle='--',
                label=f"Median = {df['cnt'].median():.1f}")
axes[0].set_xlabel('Hourly Bike Rentals (cnt)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of cnt')
axes[0].legend()

# Box plot
axes[1].boxplot(df['cnt'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_ylabel('Hourly Bike Rentals (cnt)')
axes[1].set_title('Box Plot of cnt')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

print(f"Mean:     {df['cnt'].mean():.1f}")
print(f"Median:   {df['cnt'].median():.1f}")
print(f"Std Dev:  {df['cnt'].std():.1f}")
print(f"Min:      {df['cnt'].min()}")
print(f"Max:      {df['cnt'].max()}")
print(f"Skewness: {df['cnt'].skew():.2f}")

### Hour-of-Day Pattern

The most striking feature is a **bimodal peak pattern** matching commuter behaviour:
- Morning peak at **08:00** (mean ≈ 359 rentals/hour)
- Evening peak at **17:00** (mean ≈ 462 rentals/hour — the daily maximum)
- Overnight minimum at **04:00** (mean ≈ 6 rentals/hour)

This non-linear, bimodal relationship is difficult for a linear model to capture
without feature engineering, but a Random Forest can learn it directly from the data.

In [ ]:
demand_by_hour = df.groupby('hr')['cnt'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(demand_by_hour.index, demand_by_hour.values,
        marker='o', linewidth=2, color='steelblue', markersize=5)
ax.fill_between(demand_by_hour.index, demand_by_hour.values,
                alpha=0.2, color='steelblue')

# Annotate peaks
ax.annotate(f'AM peak\n{demand_by_hour[8]:.1f}',
            xy=(8, demand_by_hour[8]),
            xytext=(9, demand_by_hour[8] + 25),
            fontsize=8, color='firebrick',
            arrowprops=dict(arrowstyle='->', color='firebrick', lw=1))
ax.annotate(f'PM peak\n{demand_by_hour[17]:.1f}',
            xy=(17, demand_by_hour[17]),
            xytext=(18, demand_by_hour[17] + 25),
            fontsize=8, color='firebrick',
            arrowprops=dict(arrowstyle='->', color='firebrick', lw=1))
ax.annotate(f'Min\n{demand_by_hour[4]:.1f}',
            xy=(4, demand_by_hour[4]),
            xytext=(5, demand_by_hour[4] + 30),
            fontsize=8, color='navy',
            arrowprops=dict(arrowstyle='->', color='navy', lw=1))

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Mean Hourly Rentals')
ax.set_title('Mean Bike Demand by Hour of Day')
ax.set_xticks(range(0, 24))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Seasonal Variation

Fall has the highest average demand (236.0); Spring has the lowest (111.1) —
a ratio of approximately **2.1:1**. This likely reflects temperature and daylight
effects on cycling.

In [ ]:
season_labels = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
demand_by_season = df.groupby('season')['cnt'].mean()

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4e9af1', '#f4a261', '#e76f51', '#6a9fb5']
bars = ax.bar(
    [season_labels[s] for s in demand_by_season.index],
    demand_by_season.values,
    color=colors,
    edgecolor='white',
    linewidth=0.8
)

for bar, val in zip(bars, demand_by_season.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Season')
ax.set_ylabel('Mean Hourly Rentals')
ax.set_title('Mean Bike Demand by Season')
ax.set_ylim(0, demand_by_season.max() * 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Weather Impact

Demand drops **monotonically** as conditions worsen:

| Code | Conditions | Mean cnt |
|------|-----------|----------|
| 1 | Clear / Few Clouds | 204.9 |
| 2 | Mist / Cloudy | 175.2 |
| 3 | Light Rain or Snow | 111.6 |
| 4 | Heavy Rain / Ice / Thunderstorm | 74.3 |

Note: Category 4 has only **3 observations** in the dataset —
its mean should be treated with caution.

In [ ]:
weather_labels = {
    1: '1: Clear',
    2: '2: Mist/Cloudy',
    3: '3: Light Rain/Snow',
    4: '4: Heavy Rain/Ice'
}
demand_by_weather = df.groupby('weathersit')['cnt'].mean()
obs_by_weather = df.groupby('weathersit')['cnt'].count()

fig, ax = plt.subplots(figsize=(8, 4))
palette = ['#4CAF50', '#FFC107', '#FF9800', '#F44336']
bars = ax.bar(
    [weather_labels[w] for w in demand_by_weather.index],
    demand_by_weather.values,
    color=palette[:len(demand_by_weather)],
    edgecolor='white',
    linewidth=0.8
)

for bar, val, n in zip(bars, demand_by_weather.values, obs_by_weather.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 2,
            f'{val:.1f}\n(n={n})',
            ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Weather Situation')
ax.set_ylabel('Mean Hourly Rentals')
ax.set_title('Mean Bike Demand by Weather Situation')
ax.set_ylim(0, demand_by_weather.max() * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Correlation Analysis

Pearson correlations between each predictor and `cnt` reveal:
- **`temp` (+0.405)** and **`atemp` (+0.401)** are the strongest linear correlates
  — warmer weather brings more riders.
- **`hr` (+0.394)** captures the time-of-day effect, though the linear coefficient
  understates the bimodal pattern.
- **`hum` (−0.323)** — higher humidity suppresses demand.
- **`weathersit` (−0.142)** — worse weather reduces demand.

**Important:** Correlations describe statistical associations, not causal mechanisms.

In [ ]:
FEATURE_COLS = [
    'season', 'yr', 'mnth', 'hr',
    'holiday', 'weekday', 'workingday', 'weathersit',
    'temp', 'atemp', 'hum', 'windspeed'
]
TARGET_COL = 'cnt'

corr_cols = FEATURE_COLS + [TARGET_COL]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True  # show lower triangle

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.5, annot_kws={'size': 7}, ax=ax
)
ax.set_title('Pearson Correlation Matrix (Predictors + cnt)')
plt.tight_layout()
plt.show()

print('\nCorrelation with target (cnt):')
print(corr_matrix[TARGET_COL].drop(TARGET_COL).sort_values(ascending=False).round(3))

---
## Section 3 — Feature Selection and Train/Test Split

### Excluded columns

| Column | Reason for exclusion |
|--------|----------------------|
| `cnt` | Target variable — not a predictor |
| `casual` | Directly contributes to `cnt` → **target leakage** |
| `registered` | Directly contributes to `cnt` → **target leakage** |
| `instant` | Row identifier — carries no predictive information |
| `dteday` | String date — calendar information already encoded in `yr`, `mnth`, `hr` |

### Feature columns used (12 predictors)

`season`, `yr`, `mnth`, `hr`, `holiday`, `weekday`, `workingday`,
`weathersit`, `temp`, `atemp`, `hum`, `windspeed`

### Chronological split

The dataset is ordered by `instant` (row number = chronological order).
We use an **80/20 chronological split** — no shuffling — so the model never
trains on data from the future relative to the test period. This is essential
for time-series-structured data to avoid information leakage.

In [ ]:
FEATURE_COLS = [
    'season', 'yr', 'mnth', 'hr',
    'holiday', 'weekday', 'workingday', 'weathersit',
    'temp', 'atemp', 'hum', 'windspeed'
]
TARGET_COL = 'cnt'

# Sort by instant to ensure chronological order
df_sorted = df.sort_values('instant').reset_index(drop=True)

# 80 / 20 chronological split
split_idx = int(len(df_sorted) * 0.80)
df_train = df_sorted.iloc[:split_idx]
df_test  = df_sorted.iloc[split_idx:]

X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET_COL]
X_test  = df_test[FEATURE_COLS]
y_test  = df_test[TARGET_COL]

print(f'Total rows:  {len(df_sorted):,}')
print(f'Train rows:  {len(df_train):,}  ({split_idx / len(df_sorted) * 100:.1f}%)')
print(f'Test rows:   {len(df_test):,}  ({len(df_test) / len(df_sorted) * 100:.1f}%)')
print()
print(f"Train date range: {df_train['dteday'].min()} -> {df_train['dteday'].max()}")
print(f"Test  date range: {df_test['dteday'].min()}  -> {df_test['dteday'].max()}")

---
## Section 4 — Baseline Model

A **DummyRegressor** (strategy=`mean`) always predicts the training-set mean count.
It has no learning, no features, and exists solely as a performance floor:
any real model must beat it to be useful.

Expected results (from verified model training):
- Baseline MAE ≈ 174.98, RMSE ≈ 232.61, R² ≈ −0.11

In [ ]:
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

mae_b  = mean_absolute_error(y_test, y_pred_baseline)
rmse_b = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
r2_b   = r2_score(y_test, y_pred_baseline)

print('=== Baseline Model (DummyRegressor  always predicts training mean) ===')
print(f'  MAE:  {mae_b:.2f}')
print(f'  RMSE: {rmse_b:.2f}')
print(f'  R2:   {r2_b:.4f}')
print()
print(f'The baseline always predicts {y_train.mean():.1f} rentals/hour.')

---
## Section 5 — Random Forest Regression

### Why Random Forest?

The EDA revealed strongly **non-linear** demand patterns:
- A bimodal hour-of-day peak (not linear)
- Weather x temperature interactions
- Year-on-year growth captured by a binary `yr` column

Random Forests handle these naturally through deep decision trees and averaging,
without requiring manual feature engineering.

### Pipeline design

1. **StandardScaler** — normalises features (not strictly necessary for trees,
   but ensures numerical consistency and reproducibility).
2. **RandomForestRegressor** — 100 trees, `random_state=42` for reproducibility.

**Preprocessing is fitted on training data only** — the scaler's mean and variance
are computed from X_train and applied to X_test without refitting.

Expected results (from verified training run):
- RF MAE ≈ 45.02, RMSE ≈ 69.62, R² ≈ 0.90

> **Note:** scikit-learn's Random Forest uses CPU parallelism (`n_jobs=-1`).
> This notebook does **not** use GPU acceleration.

In [ ]:
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

print('Fitting Random Forest pipeline (this may take 30-60 seconds)...')
rf_pipeline.fit(X_train, y_train)
print('Done.')

In [ ]:
y_pred_rf = rf_pipeline.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)

print('=== Random Forest Model ===')
print(f'  MAE:  {mae_rf:.2f}')
print(f'  RMSE: {rmse_rf:.2f}')
print(f'  R2:   {r2_rf:.4f}')

---
## Section 6 — Results and Visualizations

### Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model':  ['Baseline (DummyRegressor, mean)', 'Random Forest (100 trees, seed=42)'],
    'MAE':    [round(mae_b, 2),  round(mae_rf, 2)],
    'RMSE':   [round(rmse_b, 2), round(rmse_rf, 2)],
    'R2':     [round(r2_b, 4),   round(r2_rf, 4)],
})

print('=' * 65)
print(results.to_string(index=False))
print('=' * 65)
print()
mae_improvement = (mae_b - mae_rf) / mae_b * 100
print(f'Random Forest reduces MAE by {mae_improvement:.1f}% relative to the baseline.')

### Actual vs Predicted (Test Set)

A well-calibrated model's predictions cluster tightly along the diagonal.
Deviations from the diagonal indicate prediction errors — systematic bias
shows as a curved or offset cluster.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Random Forest ---
ax = axes[0]
ax.scatter(y_test, y_pred_rf, alpha=0.25, s=8, color='steelblue', label='Predictions')
max_val = max(y_test.max(), y_pred_rf.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual cnt')
ax.set_ylabel('Predicted cnt')
ax.set_title(f'Random Forest  Actual vs Predicted\nMAE={mae_rf:.1f}, R2={r2_rf:.3f}')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Baseline ---
ax = axes[1]
ax.scatter(y_test, y_pred_baseline, alpha=0.25, s=8, color='darkorange',
           label='Predictions')
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual cnt')
ax.set_ylabel('Predicted cnt')
ax.set_title(f'Baseline (mean)  Actual vs Predicted\nMAE={mae_b:.1f}, R2={r2_b:.3f}')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Feature Importance (Random Forest)

Feature importance from a Random Forest reflects the **mean decrease in impurity**
averaged across all trees. It indicates which features the model relied on most.

**Important:** High importance is not evidence of causation. `hr` is important because
it strongly correlates with demand patterns — but whether the hour *causes* demand
changes requires a separate causal study design.

In [ ]:
importances = rf_pipeline.named_steps['rf'].feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
mean_imp = feat_imp.mean()
colors = ['#2196F3' if v >= mean_imp else '#90CAF9' for v in feat_imp.values]
bars = ax.barh(
    feat_imp.index[::-1],
    feat_imp.values[::-1],
    color=colors[::-1],
    edgecolor='white'
)

for bar, val in zip(bars, feat_imp.values[::-1]):
    ax.text(bar.get_width() + 0.002,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=8)

ax.set_xlabel('Mean Decrease in Impurity (feature importance)')
ax.set_title('Random Forest Feature Importance\n(not causal evidence)')
ax.axvline(mean_imp, color='firebrick', linestyle='--',
           linewidth=1, label=f'Mean = {mean_imp:.3f}')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('Feature importances (sorted):')
print(feat_imp.round(4).to_string())

### Residual Analysis

Residuals = actual − predicted. A well-fitting model should have residuals
centred at zero with roughly symmetric spread.

In [ ]:
residuals_rf = y_test.values - y_pred_rf
residuals_bl = y_test.values - y_pred_baseline

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# RF residuals
axes[0].hist(residuals_rf, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='firebrick', linestyle='--', linewidth=1.5)
axes[0].axvline(residuals_rf.mean(), color='darkorange', linestyle='--',
                linewidth=1.2,
                label=f'Mean residual = {residuals_rf.mean():.1f}')
axes[0].set_xlabel('Residual (actual - predicted)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Random Forest  Residual Distribution')
axes[0].legend(fontsize=8)

# Baseline residuals
axes[1].hist(residuals_bl, bins=60, color='darkorange', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='firebrick', linestyle='--', linewidth=1.5)
axes[1].axvline(residuals_bl.mean(), color='steelblue', linestyle='--',
                linewidth=1.2,
                label=f'Mean residual = {residuals_bl.mean():.1f}')
axes[1].set_xlabel('Residual (actual - predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Baseline (mean)  Residual Distribution')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'RF residuals     Mean: {residuals_rf.mean():.2f},  Std: {residuals_rf.std():.2f}')
print(f'Baseline residuals  Mean: {residuals_bl.mean():.2f}, Std: {residuals_bl.std():.2f}')

---
## Section 7 — Key Findings and Conclusions

### 7.1 Demand Patterns (EDA findings)

**Hour-of-day** is the dominant cyclic driver of hourly demand:
- Evening commuter peak at **17:00: mean 461.5 rentals/hour**
- Morning commuter peak at **08:00: mean 359.0 rentals/hour**
- Overnight minimum at **04:00: mean 6.4 rentals/hour**

**Temperature** shows the strongest linear association with demand (`r = +0.405`).
Warm, comfortable conditions correlate with higher ridership.

**Weather** suppresses demand monotonically:
- Clear conditions: 204.9 rentals/hour
- Light rain or snow: 111.6 rentals/hour (−46% vs clear)
- Heavy rain / ice: 74.3 rentals/hour (−64% vs clear)

**Year-on-year growth:** Demand grew **63%** from 2011 (mean 143.8) to 2012 (mean 234.7),
suggesting programme expansion or increased adoption.

**Season:** Fall has the highest mean demand (236.0); Spring the lowest (111.1) —
a 2.1:1 ratio.

> All correlations are descriptive statistics for this dataset only.
> They do not establish causal mechanisms.

### 7.2 Model Performance

| Model | MAE | RMSE | R² |
|-------|-----|------|----|
| Baseline (DummyRegressor, mean) | 174.98 | 232.61 | −0.113 |
| Random Forest (100 trees, seed=42) | 45.02 | 69.62 | **0.900** |

The Random Forest achieves **R² = 0.90**, explaining 90% of the variance in
hourly demand on the held-out test set. It reduces MAE by **74.3%** relative
to the mean-prediction baseline.

Hour of day (`hr`), year (`yr`), and temperature (`temp`) are the top three
features by mean decrease in impurity.

### 7.3 Limitations

1. **Geography:** Data comes from Washington D.C. (2011–2012). Results may not
   generalise to other cities, climates, or time periods.
2. **`yr` as a predictor:** The model learns year-level growth trends. It cannot
   extrapolate to 2013 or beyond meaningfully.
3. **Weather category 4:** Only 3 observations exist in the dataset.
   Predictions under extreme conditions carry high uncertainty.
4. **No causal evidence:** Correlations and feature importances describe
   statistical patterns, not causal mechanisms.
5. **Chronological split:** The train/test boundary is a sharp date cut;
   real deployment would need ongoing retraining as conditions change.
6. **Clipping:** Predictions below zero can occur for unusual inputs; clip to 0
   before display.

### 7.4 Citation

> Fanaee-T, H., & Gama, J. (2013). Event labeling combining ensemble detectors and
> background knowledge. *Progress in Artificial Intelligence*, 2(2–3), 113–127.
> https://doi.org/10.1007/s13748-013-0040-3
>
> Dataset: UCI Machine Learning Repository — Bike Sharing Dataset (ID 275).
> https://archive.ics.uci.edu/dataset/275/bike-sharing-dataset

---
*Notebook produced as part of a multi-agent data science demonstration.*